## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

namespace FastIO {
    const int SZ = 1 << 17;
    char buf[SZ], *L = buf, *R = buf;

    inline char getcharFast() {
        if (L == R) {
            R = buf + fread(buf, 1, SZ, stdin);
            L = buf;
            if (L == R) return EOF;
        }
        return *L++;
    }

    inline int readInt() {
        int x = 0;
        char ch = getcharFast();

        while (ch < '0' || ch > '9') ch = getcharFast();

        while (ch >= '0' && ch <= '9') {
            x = x * 10 + (ch - '0');
            ch = getcharFast();
        }

        return x;
    }
}

const int MAXN = 2048;

int n, A, B;
int blockLen;
int cur[MAXN];
int wherePos[MAXN];

vector<int> answer;

inline void rebuildPosition() {
    for (int i = 0; i < n; ++i) {
        wherePos[cur[i]] = i;
    }
}

inline void useSpecialSwap() {
    answer.push_back(0);

    for (int i = 0; i < n; ++i) {
        if (cur[i] == A) {
            cur[i] = B;
        } else if (cur[i] == B) {
            cur[i] = A;
        }
    }

    rebuildPosition();
}

inline void useAdd(int delta) {
    delta %= n;
    if (delta == 0) return;

    answer.push_back(delta);

    for (int i = 0; i < n; ++i) {
        cur[i] += delta;
        if (cur[i] >= n) cur[i] -= n;
    }

    rebuildPosition();
}

inline void useXor(int mask) {
    if (mask == 0) return;

    answer.push_back(-mask);

    for (int i = 0; i < n; ++i) {
        cur[i] ^= mask;
    }

    rebuildPosition();
}

pair<int, int> getAnchorPair(int from, int to) {
    int offset = (to - from + n - blockLen + n) % n;

    int left = 0;
    int right = 0;

    for (int step = n >> 1; step >= (blockLen << 1); step >>= 1) {
        if (offset >= step) {
            offset -= step;
            right += step >> 1;
        } else {
            left += step >> 1;
        }
    }

    int lowBits = from & (blockLen - 1);

    left += n >> 1;
    left += lowBits;
    right += lowBits;

    return {left, right};
}

void exchangeValue(int u, int v) {
    int partU = (u / blockLen) & 1;
    int partV = (v / blockLen) & 1;

    if (partU == partV) {
        int bridge;

        if (partU == 0) {
            bridge = (u & (blockLen - 1)) + blockLen;
        } else {
            bridge = u & (blockLen - 1);
        }

        exchangeValue(u, bridge);
        exchangeValue(v, bridge);
        exchangeValue(u, bridge);
        return;
    }

    auto [pa, pb] = getAnchorPair(A, B);
    auto [pu, pv] = getAnchorPair(u, v);

    useAdd((pu - u + n) % n);
    useXor(pu ^ pa);
    useAdd((A - pa + n) % n);

    useSpecialSwap();

    useAdd((pa - A + n) % n);
    useXor(pu ^ pa);
    useAdd((u - pu + n) % n);
}

struct PermSolver {
    int len;
    vector<int> p;
    vector<int> seq;

    PermSolver() = default;

    PermSolver(int length, const vector<int>& source) {
        len = length;
        p.assign(source.begin(), source.begin() + length);
    }

    bool checkRange() const {
        vector<int> seen(len, 0);

        for (int x : p) {
            if (x < 0 || x >= len) return false;
            seen[x] = 1;
        }

        for (int x : seen) {
            if (!x) return false;
        }

        return true;
    }

    bool solve() {
        if (len <= 1) {
            return checkRange();
        }

        if (!checkRange()) {
            return false;
        }

        int half = len >> 1;

        vector<int> evenPart(half);
        vector<int> oddPart(half);

        for (int i = 0; i < half; ++i) {
            evenPart[i] = p[i << 1] >> 1;
            oddPart[i] = p[i << 1 | 1] >> 1;
        }

        PermSolver leftSolver(half, evenPart);
        PermSolver rightSolver(half, oddPart);

        if (!leftSolver.solve()) return false;
        if (!rightSolver.solve()) return false;

        vector<int> result;

        if (p[0] & 1) {
            result.push_back(len == 2 ? 1 : -1);
        }

        int leftMask = 0;
        for (int op : leftSolver.seq) {
            if (op > 0) {
                result.push_back(-1);
                result.push_back(1);
            } else {
                int now = op << 1;
                result.push_back(now);
                leftMask ^= (-op) << 1;
            }
        }

        if (leftMask) {
            result.push_back(-leftMask);
        }

        int rightMask = 0;
        for (int op : rightSolver.seq) {
            if (op > 0) {
                result.push_back(1);
                result.push_back(-1);
            } else {
                int now = op << 1;
                result.push_back(now);
                rightMask ^= (-op) << 1;
            }
        }

        if ((leftMask & half) != (rightMask & half)) {
            for (int i = 0; i < len / 4; ++i) {
                result.push_back(-1);
                result.push_back(1);
            }
        }

        if (leftMask >= half) leftMask -= half;
        if (rightMask >= half) rightMask -= half;

        if (leftMask != rightMask) {
            return false;
        }

        vector<int> compressed;

        for (int op : result) {
            if (!compressed.empty() && op < 0 && compressed.back() < 0) {
                int merged = (-compressed.back()) ^ (-op);
                compressed.back() = -merged;

                if (compressed.back() == 0) {
                    compressed.pop_back();
                }
            } else {
                compressed.push_back(op);
            }
        }

        seq.swap(compressed);
        return true;
    }
};

int main() {
    using FastIO::readInt;

    n = readInt();
    A = readInt();
    B = readInt();

    for (int i = 0; i < n; ++i) {
        cur[i] = readInt();
    }

    rebuildPosition();

    blockLen = (A - B + n) % n;
    blockLen &= -blockLen;

    if (blockLen == 0) {
        blockLen = n;
    }

    if (blockLen > 1) {
        vector<int> reduced(n);

        for (int i = 0; i < n; ++i) {
            reduced[i] = cur[i] & (blockLen - 1);
        }

        PermSolver solver(blockLen, reduced);

        if (!solver.solve()) {
            puts("-1");
            return 0;
        }

        for (int op : solver.seq) {
            if (op > 0) {
                useAdd(op);
            } else {
                useXor(-op);
            }
        }
    }

    for (int start = 0; start < blockLen; ++start) {
        vector<int> bucket;

        for (int pos = start; pos < n; pos += blockLen) {
            bucket.push_back(cur[pos]);
        }

        sort(bucket.begin(), bucket.end());

        bool valid = true;
        int idx = 0;

        for (int pos = start; pos < n; pos += blockLen) {
            if (bucket[idx++] != pos) {
                valid = false;
                break;
            }
        }

        if (!valid) {
            puts("-1");
            return 0;
        }

        for (int pos = start; pos < n; pos += blockLen) {
            if (cur[pos] != pos) {
                exchangeValue(pos, cur[pos]);
            }
        }
    }

    for (int i = 0; i < n; ++i) {
        if (cur[i] != i) {
            puts("-1");
            return 0;
        }
    }

    printf("%d\n", (int)answer.size());

    for (int op : answer) {
        if (op == 0) {
            puts("0");
        } else if (op < 0) {
            printf("1 %d\n", -op);
        } else {
            printf("2 %d\n", op);
        }
    }

    return 0;
}

## B 长跑

In [ ]:
import sys

def main():
    data = list(map(int, sys.stdin.read().split()))
    p = 0
    total = len(data)

    while p < total:
        if p + 3 >= total:
            break

        n = data[p]
        limit = data[p + 1]
        max_jump = data[p + 2]
        budget = data[p + 3]
        p += 4

        pos_cost = {}

        # 读站点
        for _ in range(n):
            x = data[p]
            w = data[p + 1]
            p += 2

            if x >= limit:
                continue

            if x not in pos_cost or pos_cost[x] > w:
                pos_cost[x] = w

        # 直接能到终点
        if limit <= max_jump:
            print("Yes")
            continue

        # 构建数组
        coords = [0] + sorted(pos_cost.keys())
        costs = [0] + [pos_cost[x] for x in coords[1:]]

        m = len(coords)

        dp = [10**18] * m
        dp[0] = 0

        # 用“数组模拟队列”
        queue = [0] * m
        head = 0
        tail = 0

        queue[tail] = 0
        tail += 1

        for i in range(1, m):

            # 移除不合法
            while head < tail and coords[i] - coords[queue[head]] > max_jump:
                head += 1

            if head < tail:
                dp[i] = dp[queue[head]] + costs[i]

            # 维护单调性
            while head < tail and dp[queue[tail - 1]] >= dp[i]:
                tail -= 1

            queue[tail] = i
            tail += 1

        # 找能到终点的最优
        res = 10**18

        for i in range(m - 1, -1, -1):
            if limit - coords[i] <= max_jump:
                if dp[i] < res:
                    res = dp[i]
            else:
                break

        print("Yes" if res <= budget else "No")


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
import java.io.*;
import java.util.*;

public class Main {

    static final long MOD_A = 1000000007L;
    static final long MOD_B = 1000000009L;
    static final long BASE_A = 131;
    static final long BASE_B = 13331;

    static int n;
    static String s1, s2;

    static long[] powA, powB;
    static long[] hashB1, hashB2;
    static long[] revA1, revA2;

    // 预处理 hash
    static void buildHash() {
        powA = new long[n + 2];
        powB = new long[n + 2];
        hashB1 = new long[n + 2];
        hashB2 = new long[n + 2];
        revA1 = new long[n + 2];
        revA2 = new long[n + 2];

        powA[0] = powB[0] = 1;

        for (int i = 1; i <= n; i++) {
            powA[i] = powA[i - 1] * BASE_A % MOD_A;
            powB[i] = powB[i - 1] * BASE_B % MOD_B;
        }

        for (int i = 1; i <= n; i++) {
            hashB1[i] = (hashB1[i - 1] * BASE_A + s2.charAt(i - 1)) % MOD_A;
            hashB2[i] = (hashB2[i - 1] * BASE_B + s2.charAt(i - 1)) % MOD_B;
        }

        for (int i = 1; i <= n; i++) {
            char c = s1.charAt(n - i);
            revA1[i] = (revA1[i - 1] * BASE_A + c) % MOD_A;
            revA2[i] = (revA2[i - 1] * BASE_B + c) % MOD_B;
        }
    }

    // 比较两个子串是否相等
    static boolean equalSub(int x, int y, int len) {
        int l1 = n - x + 1;
        int r1 = n - x + len;

        int l2 = y;
        int r2 = y + len - 1;

        long v1 = (revA1[r1] - revA1[l1 - 1] * powA[len]) % MOD_A;
        if (v1 < 0) v1 += MOD_A;

        long v2 = (hashB1[r2] - hashB1[l2 - 1] * powA[len]) % MOD_A;
        if (v2 < 0) v2 += MOD_A;

        if (v1 != v2) return false;

        long w1 = (revA2[r1] - revA2[l1 - 1] * powB[len]) % MOD_B;
        if (w1 < 0) w1 += MOD_B;

        long w2 = (hashB2[r2] - hashB2[l2 - 1] * powB[len]) % MOD_B;
        if (w2 < 0) w2 += MOD_B;

        return w1 == w2;
    }

    // 二分最长匹配
    static int expand(int x, int y) {
        if (x < 1 || y < 1 || x > n || y > n) return 0;

        int L = 1, R = Math.min(x, n - y + 1);
        int res = 0;

        while (L <= R) {
            int mid = (L + R) >> 1;
            if (equalSub(x, y, mid)) {
                res = mid;
                L = mid + 1;
            } else {
                R = mid - 1;
            }
        }
        return res;
    }


    static int[] palindromeRadius(String s) {
        char[] arr = new char[2 * n + 3];
        arr[0] = '@';
        arr[1] = '#';

        for (int i = 0; i < n; i++) {
            arr[2 * i + 2] = s.charAt(i);
            arr[2 * i + 3] = '#';
        }

        arr[2 * n + 2] = '$';

        int[] rad = new int[arr.length];
        int mid = 0, right = 0;

        for (int i = 1; i < arr.length - 1; i++) {
            if (i < right) {
                int mirror = 2 * mid - i;
                rad[i] = Math.min(rad[mirror], right - i);
            } else {
                rad[i] = 1;
            }

            while (arr[i + rad[i]] == arr[i - rad[i]]) {
                rad[i]++;
            }

            if (i + rad[i] > right) {
                mid = i;
                right = i + rad[i];
            }
        }

        return rad;
    }

    static int solve() {
        int[] pa = palindromeRadius(s1);
        int[] pb = palindromeRadius(s2);

        int ans = 0;

        // 奇数中心
        for (int i = 1; i <= n; i++) {
            int r1 = (pa[2 * i] - 2) / 2;
            ans = Math.max(ans, 2 * r1 + 1 + 2 * expand(i - r1 - 1, i + r1));

            int r2 = (pb[2 * i] - 2) / 2;
            ans = Math.max(ans, 2 * r2 + 1 + 2 * expand(i - r2, i + r2 + 1));
        }

        // 偶数中心
        for (int i = 0; i <= n; i++) {
            int r1 = (pa[2 * i + 1] - 1) / 2;
            ans = Math.max(ans, 2 * r1 + 2 * expand(i - r1, i + r1));

            int r2 = (pb[2 * i + 1] - 1) / 2;
            ans = Math.max(ans, 2 * r2 + 2 * expand(i - r2 + 1, i + r2 + 1));
        }

        return ans;
    }

    public static void main(String[] args) throws Exception {
        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));
        String line;

        while ((line = br.readLine()) != null) {
            line = line.trim();
            if (line.length() == 0) continue;

            n = Integer.parseInt(line);
            s1 = br.readLine().trim();
            s2 = br.readLine().trim();

            buildHash();
            System.out.println(solve());
        }
    }
}

## D 优惠券

In [ ]:
import java.io.*;
import java.util.*;

public class Main {

    static final int MAXN = 1000005;

    static int[] state = new int[MAXN];
    static int[] last = new int[MAXN];
    static int[] vis = new int[MAXN];
    static int round = 0;

    static void init(int x) {
        if (vis[x] != round) {
            vis[x] = round;
            state[x] = 0;
            last[x] = 0;
        }
    }

    public static void main(String[] args) {
        FastScanner fs = new FastScanner();
        PrintWriter out = new PrintWriter(System.out);

        while (true) {
            String t = fs.next();
            if (t == null) break;

            int m = Integer.parseInt(t);
            round++;

            TreeSet<Integer> set = new TreeSet<>();

            boolean ok = true;
            int fail = -1;

            for (int i = 1; i <= m; i++) {

                String s = fs.next();
                if (s == null) break;

                char c = s.charAt(0);
                int x = 0;
                boolean isQ = false;

                if (c == 'I' || c == 'O') {
                    if (s.length() > 1) {
                        x = Integer.parseInt(s.substring(1));
                    } else {
                        x = Integer.parseInt(fs.next());
                    }
                } else {
                    isQ = true;

                    // 清理多余数字
                    while (true) {
                        String p = fs.peek();
                        if (p == null) break;
                        try {
                            Integer.parseInt(p);
                            fs.next();
                        } catch (Exception e) {
                            break;
                        }
                    }
                }

                if (!ok) continue;

                if (isQ) {
                    set.add(i);
                    continue;
                }

                init(x);

                int need = (c == 'I') ? 1 : 0;

                if (state[x] == need) {
                    Integer pos = set.higher(last[x]); // 找 > last[x] 的最小位置

                    if (pos == null || pos >= i) {
                        ok = false;
                        fail = i;
                    } else {
                        set.remove(pos);
                        last[x] = i;
                    }
                } else {
                    state[x] = need;
                    last[x] = i;
                }
            }

            out.println(ok ? -1 : fail);
        }

        out.flush();
    }

    // 简化输入
    static class FastScanner {
        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));
        StringTokenizer st;
        String cache;

        String next() {
            if (cache != null) {
                String t = cache;
                cache = null;
                return t;
            }
            try {
                while (st == null || !st.hasMoreTokens()) {
                    String line = br.readLine();
                    if (line == null) return null;
                    st = new StringTokenizer(line);
                }
                return st.nextToken();
            } catch (IOException e) {
                return null;
            }
        }

        String peek() {
            if (cache == null) cache = next();
            return cache;
        }
    }
}

## E 任意点

In [ ]:
import sys

def solve():
    # 读取所有输入数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    n = int(input_data[0])
    points = []
    idx = 1

    # 解析坐标点
    for _ in range(n):
        x = int(input_data[idx])
        y = int(input_data[idx+1])
        points.append((x, y))
        idx += 2

    visited = [False] * n

    # DFS 遍历连通块
    def dfs(u):
        visited[u] = True
        for v in range(n):
            if not visited[v]:
                # 如果横坐标或纵坐标相同，说明它们属于同一个连通块
                if points[u][0] == points[v][0] or points[u][1] == points[v][1]:
                    dfs(v)

    components = 0

    # 计算连通块的数量
    for i in range(n):
        if not visited[i]:
            components += 1
            dfs(i)

    # 需要添加的点数 = 连通块数量 - 1
    print(components - 1)

if __name__ == '__main__':
    solve()

## F 通配符匹配

In [ ]:
import java.io.*;
import java.util.*;

public class Main {
    public static void main(String[] args) throws IOException {
        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));
        PrintWriter out = new PrintWriter(System.out);

        String pattern = br.readLine();
        if (pattern == null) return;
        pattern = pattern.trim();

        String nStr = br.readLine();
        if (nStr == null) return;
        int n = Integer.parseInt(nStr.trim());

        // 预处理双哈希的 Base 的幂次方
        long[] P1 = new long[100005];
        long[] P2 = new long[100005];
        P1[0] = 1; P2[0] = 1;
        for (int i = 1; i <= 100000; i++) {
            P1[i] = (P1[i - 1] * 131) % 1000000007L;
            P2[i] = (P2[i - 1] * 13331) % 1000000009L;
        }

        // 将模式串分割为 Chunk
        List<Chunk> chunks = new ArrayList<>();
        StringBuilder sb = new StringBuilder();
        for (int i = 0; i < pattern.length(); i++) {
            char c = pattern.charAt(i);
            if (c == '*' || c == '?') {
                if (sb.length() > 0) {
                    chunks.add(new Chunk(0, sb.toString()));
                    sb.setLength(0);
                }
                if (c == '*') {
                    // 合并连续的 '*'
                    if (chunks.isEmpty() || chunks.get(chunks.size() - 1).type != 2) {
                        chunks.add(new Chunk(2, ""));
                    }
                } else {
                    chunks.add(new Chunk(1, ""));
                }
            } else {
                sb.append(c);
            }
        }
        if (sb.length() > 0) {
            chunks.add(new Chunk(0, sb.toString()));
        }

        int m = chunks.size();
        boolean[][] dp = new boolean[m + 1][100005];
        long[] H1 = new long[100005];
        long[] H2 = new long[100005];

        for (int i = 0; i < n; i++) {
            String s = br.readLine();
            if (s == null) break;
            s = s.trim();
            int L = s.length();

            // 计算当前字符串的前缀双哈希
            for (int j = 1; j <= L; j++) {
                H1[j] = (H1[j - 1] * 131 + s.charAt(j - 1)) % 1000000007L;
                H2[j] = (H2[j - 1] * 13331 + s.charAt(j - 1)) % 1000000009L;
            }

            // 初始化 DP 表
            for(int cIdx = 0; cIdx <= m; cIdx++) {
                Arrays.fill(dp[cIdx], 0, L + 1, false);
            }

            // 边界条件：模式串末尾可以匹配字符串末尾
            dp[m][L] = true;

            // 逆向递推 DP
            for (int cIdx = m - 1; cIdx >= 0; cIdx--) {
                Chunk c = chunks.get(cIdx);
                for (int sIdx = L; sIdx >= 0; sIdx--) {
                    if (c.type == 2) { // 当前块是 '*'
                        dp[cIdx][sIdx] = dp[cIdx + 1][sIdx];
                        if (sIdx < L) {
                            dp[cIdx][sIdx] |= dp[cIdx][sIdx + 1];
                        }
                    } else if (c.type == 1) { // 当前块是 '?'
                        if (sIdx < L) {
                            dp[cIdx][sIdx] = dp[cIdx + 1][sIdx + 1];
                        }
                    } else { // 当前块是普通字符串
                        int len = c.len;
                        if (sIdx + len <= L) {
                            // O(1) 获取子串的哈希值
                            long h1 = (H1[sIdx + len] - H1[sIdx] * P1[len]) % 1000000007L;
                            if (h1 < 0) h1 += 1000000007L;
                            long h2 = (H2[sIdx + len] - H2[sIdx] * P2[len]) % 1000000009L;
                            if (h2 < 0) h2 += 1000000009L;
                            long hash = (h1 << 32) | h2;

                            // 哈希值比较
                            if (hash == c.hash) {
                                dp[cIdx][sIdx] = dp[cIdx + 1][sIdx + len];
                            }
                        }
                    }
                }
            }

            if (dp[0][0]) {
                out.println("YES");
            } else {
                out.println("NO");
            }
        }
        out.flush();
    }

    static class Chunk {
        int type; // 0 = 字符串, 1 = '?', 2 = '*'
        int len;
        long hash;

        Chunk(int type, String s) {
            this.type = type;
            this.len = s.length();
            if (type == 0) {
                long h1 = 0, h2 = 0;
                for (int i = 0; i < s.length(); i++) {
                    h1 = (h1 * 131 + s.charAt(i)) % 1000000007L;
                    h2 = (h2 * 13331 + s.charAt(i)) % 1000000009L;
                }
                // 使用 long 的高低 32 位合并两个哈希值避免哈希碰撞
                this.hash = (h1 << 32) | h2;
            }
        }
    }
}

## G 汉诺塔

In [ ]:
import sys

def solve():
    # 一次性读取所有输入，避免错位
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    n = int(input_data[0])
    # 获取 6 个优先级的字符串排列
    priorities = input_data[1:7]

    # 构建优先级字典，索引越小优先级越高
    pri_map = {move: idx for idx, move in enumerate(priorities)}

    # dp[i][x] 表示将 i 个盘子从柱子 x 移走所需要的步数
    dp = [[0] * 3 for _ in range(n + 1)]
    # to[i][x] 表示将 i 个盘子从柱子 x 移走，最终会落在哪根柱子上
    to = [[0] * 3 for _ in range(n + 1)]

    pegs = ['A', 'B', 'C']

    # ---------------- 初始化 i = 1 的情况 ----------------
    for x in range(3):
        best_y = -1
        best_pri = 100
        for y in range(3):
            if x == y:
                continue
            move = pegs[x] + pegs[y]
            # 寻找从 x 出发，优先级最高的目标柱子 y
            if pri_map[move] < best_pri:
                best_pri = pri_map[move]
                best_y = y
        to[1][x] = best_y
        dp[1][x] = 1

    # ---------------- 状态转移 i >= 2 ----------------
    for i in range(2, n + 1):
        for x in range(3):
            y = to[i-1][x]     # i-1 个盘子第一步会去的位置
            z = 3 - x - y      # 剩下的那根空柱子 (0+1+2=3, 故用 3-x-y 快速求出第三根柱子)

            w = to[i-1][y]     # i-1 个盘子第二步会去的位置

            if w == z:
                # 情况 A：i-1 个盘子顺势盖在了最大盘子上
                dp[i][x] = dp[i-1][x] + 1 + dp[i-1][y]
                to[i][x] = z
            elif w == x:
                # 情况 B：i-1 个盘子退回了起点，最大盘子被迫走第二步
                dp[i][x] = dp[i-1][x] + 1 + dp[i-1][y] + 1 + dp[i-1][x]
                to[i][x] = y

    # 题目要求的是把所有盘子从 A 柱 (索引 0) 移走所需要的步数
    print(dp[n][0])

if __name__ == '__main__':
    solve()

## H 马步距离

In [ ]:
import sys

def solve():
    # 读取所有输入，忽略多余空白符
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    # 获取四个坐标
    xp = int(input_data[0])
    yp = int(input_data[1])
    xs = int(input_data[2])
    ys = int(input_data[3])

    # 计算绝对距离差
    dx = abs(xp - xs)
    dy = abs(yp - ys)

    # 利用对称性，始终保持 dx 是较长的一边
    if dx < dy:
        dx, dy = dy, dx

    # 处理公式失效的两个特例
    if dx == 1 and dy == 0:
        print(3)
    elif dx == 2 and dy == 2:
        print(4)
    else:
        # 使用基础贪心公式计算最少步数
        ans = max((dx + 1) // 2, (dx + dy + 2) // 3)

        # 校准奇偶性
        if ans % 2 != (dx + dy) % 2:
            ans += 1

        print(ans)

if __name__ == '__main__':
    solve()

## I 直方图最大矩形

In [ ]:
from typing import List

class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        # 在末尾添加一个高度为 0 的哨兵，强制最后清空栈内所有元素
        heights.append(0)
        stack = []  # 单调递增栈，存储柱子的索引
        max_area = 0

        for i in range(len(heights)):
            # 如果当前柱子比栈顶柱子矮，说明找到了栈顶柱子的右边界
            while stack and heights[i] < heights[stack[-1]]:
                # 弹出栈顶元素作为当前计算矩形的高
                h = heights[stack.pop()]

                # 计算宽：
                # 如果栈为空，说明弹出的柱子向左可以延伸到最左端 (索引 0)
                # 否则，向左延伸到新栈顶元素的右侧
                w = i if not stack else i - stack[-1] - 1

                # 更新最大面积
                max_area = max(max_area, h * w)

            # 当前元素入栈
            stack.append(i)

        return max_area

## J 消防局的设立

In [ ]:
import sys

def solve():
    # 读取所有的输入参数
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    n = int(input_data[0])

    # 特判 n 为极小值的情况
    if n == 0:
        print(0)
        return
    if n == 1:
        print(1)
        return

    # 构建树的邻接表
    adj = [[] for _ in range(n + 1)]
    for i in range(2, n + 1):
        u = i
        v = int(input_data[i - 1])
        adj[u].append(v)
        adj[v].append(u)

    # BFS 建立纯正的父子层级关系，同时获取逆向的拓扑序
    children = [[] for _ in range(n + 1)]
    queue = [1]
    head = 0
    visited = [False] * (n + 1)
    visited[1] = True

    while head < len(queue):
        u = queue[head]
        head += 1
        for v in adj[u]:
            if not visited[v]:
                visited[v] = True
                children[u].append(v)
                queue.append(v)

    # 逆向遍历 (从叶子节点往根节点收缩)
    nodes_sorted = queue[::-1]

    dist_station = [float('inf')] * (n + 1)
    dist_uncov = [0] * (n + 1)
    ans = 0

    for u in nodes_sorted:
        d_st = float('inf')
        d_unc = 0  # 初始为0，代表 u 自己是一个未被覆盖的节点

        # 收集子节点的状态
        for v in children[u]:
            if dist_station[v] + 1 < d_st:
                d_st = dist_station[v] + 1
            if dist_uncov[v] + 1 > d_unc:
                d_unc = dist_uncov[v] + 1

        # 判断子树内的消防局能否覆盖子树内最远的未覆盖节点
        if d_st + d_unc <= 2:
            d_unc = float('-inf')

        # 贪心策略：如果最远未覆盖距离达到 2，不能再等了，强制在当前 u 建立消防局
        if d_unc == 2:
            ans += 1
            d_st = 0
            d_unc = float('-inf')

        dist_station[u] = d_st
        dist_uncov[u] = d_unc

    # 如果根节点范围内还有未被覆盖的节点，额外增加 1 个消防局
    if dist_uncov[1] >= 0:
        ans += 1

    print(ans)

if __name__ == '__main__':
    solve()